# Word2Vec Tasks
## Completed by William Brannock (svv8fs)

This notebook contains the word2vec tasks for the Final Project. This notebook was guided by this class notebook. https://www.kaggle.com/code/ontoligent/uva-ds-5001-m09-word2vec-with-wine-reviews

# Set Up

In [1]:
import contextlib
import io
import warnings

import gensim
import pandas as pd
import plotly.express as px
import plotly.io as pio
from gensim.corpora import Dictionary
from gensim.models import word2vec
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore", category=FutureWarning)
pio.renderers.default = "plotly_mimetype+notebook_connected"

In [2]:
gensim.__version__

'4.4.0'

# Config

In [3]:
data_home = "."
output_dir = "output"
data_prefix = "constitutions"
source_dir = output_dir
CSV_DELIM = "|"

OHCO = ["constitution_id", "provision_num", "para_num", "sent_num", "token_num"]
BAG = OHCO[:4]
bag_name = "SENTS"

vector_size = 200
min_count = 50
random_state = 23

# Import Tables

In [4]:
# Read in corpus
CORPUS = pd.read_csv(
    f"{source_dir}/{data_prefix}-CORPUS.csv",
    sep=CSV_DELIM,
    keep_default_na=False,
    usecols=OHCO + ["term_str"],
)

CORPUS.shape

(4296300, 6)

# Prepare Sentence Bags


In [5]:
TOKENS = CORPUS[CORPUS.term_str != ""].copy()

docs_by_bag = TOKENS.groupby(BAG, sort=False).term_str.apply(list)
docs = [terms for terms in docs_by_bag.tolist() if len(terms) > 1]



In [6]:
docs[:5]

[['in',
  'the',
  'name',
  'of',
  'allah',
  'the',
  'most',
  'beneficent',
  'the',
  'most',
  'merciful'],
 ['praise',
  'be',
  'to',
  'allah',
  'the',
  'cherisher',
  'and',
  'sustainer',
  'of',
  'worlds',
  'and',
  'praise',
  'and',
  'peace',
  'be',
  'upon',
  'mohammad',
  'his',
  'last',
  'messenger',
  'and',
  'his',
  'disciples',
  'and',
  'followers'],
 ['we', 'the', 'people', 'of', 'afghanistan'],
 ['believing',
  'firmly',
  'in',
  'almighty',
  'god',
  'relying',
  'on',
  'his',
  'divine',
  'will',
  'and',
  'adhering',
  'to',
  'the',
  'holy',
  'religion',
  'of',
  'islam'],
 ['realizing',
  'the',
  'previous',
  'injustices',
  'miseries',
  'and',
  'innumerable',
  'disasters',
  'which',
  'have',
  'befallen',
  'our',
  'country']]

# Generate Word Embeddings

In [7]:
dictionary = Dictionary(docs)

w2v_params = dict(
    vector_size=vector_size,
    window=2,
    min_count=min_count,
    workers=4,
)

stderr_buffer = io.StringIO()
with contextlib.redirect_stderr(stderr_buffer):
    model = word2vec.Word2Vec(docs, **w2v_params)

model

# VOCAB_W2V

In [8]:
vector_cols = [f"w2v_{i:03d}" for i in range(model.wv.vector_size)]

WV = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key, columns=vector_cols)
WV.index.name = "term_str"

VOCAB_W2V = WV.copy()
VOCAB_W2V.head()

,w2v_000,w2v_001,w2v_002,w2v_003,w2v_004,w2v_005,w2v_006,w2v_007,w2v_008,w2v_009,...,w2v_190,w2v_191,w2v_192,w2v_193,w2v_194,w2v_195,w2v_196,w2v_197,w2v_198,w2v_199
term_str,,,,,,,,,,,,,,,,,,,,,
the,-0.151566,0.701431,-0.574343,0.011376,-0.278563,0.080294,0.245978,0.139016,-0.475719,-0.180672,...,-0.056717,-0.457708,-0.575757,-0.177010,-0.006457,-0.658492,-0.213012,0.356814,-0.558356,0.572237
of,0.091819,-0.118065,-0.183558,0.529104,-0.339020,0.204121,-0.194797,-0.100469,-0.271202,0.041506,...,-0.435708,-0.116900,0.121334,-0.384459,0.597033,0.384040,-0.956567,-0.224010,-1.629509,-0.156791
and,-0.334713,0.489464,-0.811356,0.250539,0.762163,0.227519,0.266575,0.210651,0.534860,0.040874,...,-0.162232,-0.371885,0.098749,0.374683,0.405226,0.667919,0.014731,0.110128,0.082469,0.148659
to,0.938660,0.274651,0.086720,0.767844,0.027300,1.693668,1.332623,-0.121951,-0.333108,-0.107295,...,-1.223299,-0.931258,-0.424670,0.787881,0.086506,0.662061,0.070381,0.588764,1.006335,-0.182099
in,-0.200408,0.024829,0.137012,0.346861,0.416209,1.137643,0.136753,0.107022,-0.390802,-0.176132,...,-0.824901,0.748625,0.800138,0.344238,1.342467,0.267629,-0.761403,0.426886,-0.716416,-0.423480


# t-SNE Plot

In [9]:
tsne_vectors = VOCAB_W2V[vector_cols]
perplexity = 40

tsne_engine = TSNE(
    perplexity=perplexity,
    n_components=2,
    init="pca",
    max_iter=2500,
    random_state=random_state,
    learning_rate="auto",
)

TSNE_W2V = pd.DataFrame(
    tsne_engine.fit_transform(tsne_vectors),
    columns=["x", "y"],
    index=tsne_vectors.index,
)
TSNE_W2V.index.name = "term_str"
TSNE_W2V.head()

,x,y
term_str,,
the,9.934578,10.647046
of,24.546600,-2.821373
and,22.279564,-33.718098
to,2.775492,11.181621
in,-26.608044,8.123830


In [10]:
fig = px.scatter(
    TSNE_W2V.reset_index(),
    x="x",
    y="y",
    hover_name="term_str",
    text="term_str",
    height=900,
    width=1100,
    title="Word2Vec Term Embeddings Projected with t-SNE",
    template="plotly_white",
)
fig.update_traces(marker={"opacity": 0.65, "line": {"width": 0.4, "color": "#333333"}})
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="t-SNE 1", yaxis_title="t-SNE 2")
fig.show()

# Save

In [11]:
save_path = f"{output_dir}/{data_prefix}"

VOCAB_W2V.to_csv(f"{save_path}-VOCAB_W2V-{bag_name}.csv", sep=CSV_DELIM)